# Frequentist Engagement Analysis

Primary analysis for the tweet A/B test using engagement per tweet.


In [ ]:
import pandas as pd
from scipy import stats

RESULTS_PATH = "data/experiment_results.csv"
METRIC_COLUMNS = ["likes", "replies", "reposts", "bookmarks"]

df = pd.read_csv(RESULTS_PATH)
df["engagement"] = df[METRIC_COLUMNS].sum(axis=1)
df.head()


In [ ]:
summary = (
    df.groupby("variant")["engagement"]
    .agg(count="count", mean="mean", std="std", median="median")
    .round(3)
)
summary


In [ ]:
A = df.loc[df["variant"] == "A", "engagement"]
B = df.loc[df["variant"] == "B", "engagement"]

t_stat, p_value = stats.ttest_ind(
    A,
    B,
    equal_var=False,
    nan_policy="raise",
)

mean_diff = B.mean() - A.mean()
pooled_std = (((A.std(ddof=1) ** 2) + (B.std(ddof=1) ** 2)) / 2) ** 0.5
cohens_d = mean_diff / pooled_std

print("A mean:", round(A.mean(), 3))
print("B mean:", round(B.mean(), 3))
print("Mean difference (B - A):", round(mean_diff, 3))
print("Welch t-statistic:", round(t_stat, 3))
print("p-value:", round(p_value, 4))
print("Cohen's d:", round(cohens_d, 3))


In [ ]:
def mean_confidence_interval(values, confidence=0.95):
    values = pd.Series(values).dropna()
    n = len(values)
    mean = values.mean()
    se = stats.sem(values)
    margin = se * stats.t.ppf((1 + confidence) / 2, n - 1)
    return mean - margin, mean + margin

intervals = {
    variant: mean_confidence_interval(group["engagement"])
    for variant, group in df.groupby("variant")
}

pd.DataFrame(intervals, index=["ci_lower", "ci_upper"]).T.round(3)


In [ ]:
alpha = 0.05
if p_value < alpha:
    print("Result: statistically significant difference at alpha=0.05.")
else:
    print("Result: no statistically significant difference at alpha=0.05.")

if mean_diff > 0:
    print("Direction: B has higher sample mean engagement than A.")
elif mean_diff < 0:
    print("Direction: A has higher sample mean engagement than B.")
else:
    print("Direction: A and B have equal sample mean engagement.")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.boxplot(data=df, x="variant", y="engagement")
plt.title("Tweet Engagement A/B Test")
plt.xlabel("Variant")
plt.ylabel("Engagement")
plt.show()
